# Pattern 3: Inline Claim-Based Authorization (Agent-Side)

The agent reads the user's JWT claims (role, department) and uses them to construct narrower tool calls. For example, it adds a department filter for managers. The backend service still only receives the API key -- it has no user identity.

In practice, this claim-reading and narrowing would typically live in the agent itself -- in its system prompt or tool-calling logic. We implement it in the MCP server's AuthHandler because that gives us a visible code hook, but the effect is the same: the service is blind to the narrowing.

**What the service sees:** API key only. Same as pattern 1.

**Weakness:** Authz is scattered in agent-side code, hard to audit, and bypassable if the agent or MCP server is compromised. The service has no way to verify that narrowing occurred.

In [1]:
from framework.runner import PatternRunner
runner = PatternRunner("p03_inline_claim_agent")

## The auth code

This is the lesson. Read both files to understand what happens at each boundary.

In [2]:
runner.show_auth_code()

╭──────────────────────── p03_inline_claim_agent/mcp_auth.py ────────────────────────╮
│    1 """Pattern 3: Inline Claim-Based Authorization (agent-side).                  │
│    2                                                                               │
│    3 The agent reads the user's JWT claims (role, department) and uses them to     │
│    4 construct narrower tool calls -- e.g. adding a department filter for          │
│    5 managers. The service still only receives the API key; it has no user         │
│    6 identity. The authz logic lives at the agent layer.                           │
│    7                                                                               │
│    8 In practice, this narrowing would typically live in the agent itself -- in    │
│    9 its system prompt, tool-calling logic, or a pre-processing step that reads    │
│   10 claims and decides how to parameterize each tool call. We implement it here   │
│   11 in the MCP server's AuthHandler because that's where we have a code hook      │
│   12 in this teaching repo, but the effect is the same: the service is blind to    │
│   13 the narrowing and has no way to verify it happened.                           │
│   14                                                                               │
│   15 This is the pattern's weakness: authz is scattered in agent-side code,        │
│   16 hard to audit, and bypassable if the agent or MCP server is compromised.      │
│   17 """                                                                           │
│   18                                                                               │
│   19 from framework.auth_helpers import decode_jwt                                 │
│   20 from framework.config import SHARED_SERVICE_API_KEY                           │
│   21 from framework.mcp.auth import AuthHandler                                    │
│   22                                                                               │
│   23                                                                               │
│   24 class InlineClaimAgentHandler(AuthHandler):                                   │
│   25     """Reads JWT claims and narrows tool calls before they reach the service. │
│   26                                                                               │
│   27     In a real deployment, this logic would more likely live in the agent      │
│   28     itself (e.g. the LLM is instructed to scope queries based on the          │
│   29     user's role and department). We put it here to make it visible in         │
│   30     code, but the pedagogical point is the same: the service never sees       │
│   31     the JWT, only the narrowed query.                                         │
│   32     """                                                                       │
│   33                                                                               │
│   34     def __init__(self):                                                       │
│   35         self._last_extra_params: dict | None = None                           │
│   36                                                                               │
│   37     async def prepare_request(self, user_context, headers):                   │
│   38         headers["X-API-Key"] = SHARED_SERVICE_API_KEY                         │
│   39                                                                               │
│   40         jwt = user_context.get("jwt")                                         │
│   41         if not jwt:                                                           │
│   42             return headers                                                    │
│   43                                                                               │
│   44         # Read claims from the JWT to narrow the tool call                    │
│   45         claims = decode_jwt(jwt)                                              

╭────────────────────── p03_inline_claim_agent/service_auth.py ───────────────────────╮
│    1 """Pattern 3: Inline Claim-Based Authorization (service side).                 │
│    2                                                                                │
│    3 The service only sees the API key. It has no user identity -- all the claim    │
│    4 reading and narrowing happened at the MCP server before the request arrived.   │
│    5 This is intentionally identical to pattern 1's service auth.                   │
│    6 """                                                                            │
│    7                                                                                │
│    8 from fastapi import Request                                                    │
│    9 from framework.services.identity import Identity                               │
│   10                                                                                │
│   11 EXPECTED_API_KEY = "dev-shared-api-key"                                        │
│   12                                                                                │
│   13                                                                                │
│   14 async def get_expense_identity(request: Request) -> Identity:                  │
│   15     api_key = request.headers.get("x-api-key")                                 │
│   16     if not api_key:                                                            │
│   17         return Identity(method="none", detail="no auth provided")              │
│   18     if api_key == EXPECTED_API_KEY:                                            │
│   19         return Identity(                                                       │
│   20             method="api_key",                                                  │
│   21             detail="shared service credential, no user identity",              │
│   22         )                                                                      │
│   23     return Identity(                                                           │
│   24         method="none",                                                         │
│   25         detail=f"X-API-Key did not match (received prefix: {api_key[:8]}...)", │
│   26     )                                                                          │
│   27                                                                                │
│   28                                                                                │
│   29 get_document_identity = get_expense_identity                                   │
│   30                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────╯

The mcp_auth.py decodes the JWT claims and adds narrowing query params (e.g. department filter for managers). But it only sends the API key to the service -- not the JWT itself.

In a real-world agent, this logic would more likely live in the agent's system prompt or tool-calling code ("if the user is a manager, filter expenses by their department"). We put it in the MCP server here to make it visible as code you can read, but the point is the same: the narrowing happens before the request reaches the service.

Compare the service_auth.py: it's identical to pattern 1. The service has no idea that claim-based narrowing happened.

In [3]:
await runner.start()

[04/14/26 06:20:46] INFO     StreamableHTTP session manager started                  ]8;id=818190;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=151404;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     StreamableHTTP session manager started                  ]8;id=789087;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=385265;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:49497/mcp "HTTP/1.1 200 OK"        ]8;id=559006;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=588659;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=456174;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=897039;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

                    INFO     Terminating session: None                                       ]8;id=927372;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=272302;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

                    INFO     HTTP Request: POST http://127.0.0.1:49497/mcp "HTTP/1.1 200 OK"        ]8;id=250661;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=219934;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:49497/mcp "HTTP/1.1 200 OK"        ]8;id=663179;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=767735;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:49498/mcp "HTTP/1.1 200 OK"        ]8;id=164046;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=954905;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=902708;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=503658;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=619559;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=863276;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

## Run scenarios

Let's see this pattern in action with different users.

In [4]:
await runner.run_as("alice", "What are my expenses?")

[04/14/26 06:20:47] INFO     HTTP Request: POST                                                     ]8;id=149246;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=743193;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭───────────────────────────────╮
│ [alice] What are my expenses? │
╰───────────────────────────────╯

                    INFO     Terminating session: None                                       ]8;id=516302;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=406349;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

                    INFO     Processing request of type ListToolsRequest                              ]8;id=988860;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=47015;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     Terminating session: None                                       ]8;id=477139;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=10132;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

[04/14/26 06:20:52] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=583048;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=702262;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:21:06] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=425225;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=161515;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "api_key",                   │
│     │              │      │        │ "identity_detail": "shared service credential, no user identity", "count": │
│     │              │      │        │ 8, "expenses": [{"id": 1, "user_id": "alice", "department": "engineering", │
│     │              │      │        │ "amount": 42.5, "category": "software", "description": "JetBrains AI       │
│     │              │      │        │ assistant subscription",                                                   │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are the expenses for the current account (note: this session uses a shared service credential, not tied to │
│ a specific person).                                                                                             │
│                                                                                                                 │
│ Summary                                                                                                         │
│ - Total: 3344.50                                                                                                │
│ - Approved: 1894.50                                                                                             │
│ - Pending: 1450.00                                                                                              │
│ - Count: 8                                                                                                      │
│                                                                                                                 │
│ Expenses                                                                                                        │
│ - ID 1: user alice | department engineering | category software | 42.50 | JetBrains AI assistant subscription | │
│ approved                                                                                                        │
│ - ID 2: user alice | department engineering | category travel | 156.00 | Train ticket to client offsite |       │
│ approved                                                                                                        │
│ - ID 3: user alice | department engineering | category books | 89.00 | Designing Data-Intensive Applications |  │
│ approved                                                                                                        │
│ - ID 4: user alice | department engineering | category hardware | 1450.00 | External 4K monitor | pending       │
│ - ID 5: user bob | department engineering | category training | 320.00 | OAuth 2.0 deep-dive workshop |         │
│ approved                                                                                                        │
│ - ID 6: user bob | department engineering | category meals | 67.00 | Team lunch after the migration shipped |   │
│ approved                                                                                                        │
│ - ID 7: user dave | department platform | category training | 980.00 | KubeCon ticket | approved                │
│ - ID 8: user dave | department platform | category software | 240.00 | Datadog license seat | approved          │
│                                                                                                                 │
│ Would you like me to filter by department, status, or show only expenses tied to a specific user once your      │
│ identity is linked?                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are the expenses for the current account (note: this session uses a shared service credential, not tied to a specific person).\n\nSummary\n- Total: 3344.50\n- Approved: 1894.50\n- Pending: 1450.00\n- Count: 8\n\nExpenses\n- ID 1: user alice | department engineering | category software | 42.50 | JetBrains AI assistant subscription | approved\n- ID 2: user alice | department engineering | category travel | 156.00 | Train ticket to client offsite | approved\n- ID 3: user alice | department engineering | category books | 89.00 | Designing Data-Intensive Applications | approved\n- ID 4: user alice | department engineering | category hardware | 1450.00 | External 4K monitor | pending\n- ID 5: user bob | department engineering | category training | 320.00 | OAuth 2.0 deep-dive workshop | approved\n- ID 6: user bob | department engineering | category meals | 67.00 | Team lunch after the migration shipped | approved\n- ID 7: user dave | department platform | category t

In [5]:
await runner.run_as("bob", "Show me all expenses")

                    INFO     HTTP Request: POST                                                     ]8;id=512019;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=663802;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭────────────────────────────╮
│ [bob] Show me all expenses │
╰────────────────────────────╯

[04/14/26 06:21:07] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=940337;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=431964;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:21:09] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=323273;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=357738;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=281594;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=434638;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

[04/14/26 06:21:13] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=704232;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=198899;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:21:22] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=250295;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=806603;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "api_key",                   │
│     │              │      │        │ "identity_detail": "shared service credential, no user identity", "count": │
│     │              │      │        │ 6, "expenses": [{"id": 1, "user_id": "alice", "department": "engineering", │
│     │              │      │        │ "amount": 42.5, "category": "software", "description": "JetBrains AI       │
│     │              │      │        │ assistant subscription",                                                   │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are your expenses (6 total, sum: $2,124.50):                                                               │
│                                                                                                                 │
│ 1) ID 1 | user: alice | dept: engineering | amount: $42.50 | category: software | description: JetBrains AI     │
│ assistant subscription | status: approved                                                                       │
│ 2) ID 2 | user: alice | dept: engineering | amount: $156.00 | category: travel | description: Train ticket to   │
│ client offsite | status: approved                                                                               │
│ 3) ID 3 | user: alice | dept: engineering | amount: $89.00 | category: books | description: Designing           │
│ Data-Intensive Applications | status: approved                                                                  │
│ 4) ID 4 | user: alice | dept: engineering | amount: $1,450.00 | category: hardware | description: External 4K   │
│ monitor | status: pending                                                                                       │
│ 5) ID 5 | user: bob | dept: engineering | amount: $320.00 | category: training | description: OAuth 2.0         │
│ deep-dive workshop | status: approved                                                                           │
│ 6) ID 6 | user: bob | dept: engineering | amount: $67.00 | category: meals | description: Team lunch after the  │
│ migration shipped | status: approved                                                                            │
│                                                                                                                 │
│ Would you like to:                                                                                              │
│ - filter by department or status,                                                                               │
│ - export these,                                                                                                 │
│ - or approve the pending expense (ID 4) if you have permission?                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are your expenses (6 total, sum: $2,124.50):\n\n1) ID 1 | user: alice | dept: engineering | amount: $42.50 | category: software | description: JetBrains AI assistant subscription | status: approved\n2) ID 2 | user: alice | dept: engineering | amount: $156.00 | category: travel | description: Train ticket to client offsite | status: approved\n3) ID 3 | user: alice | dept: engineering | amount: $89.00 | category: books | description: Designing Data-Intensive Applications | status: approved\n4) ID 4 | user: alice | dept: engineering | amount: $1,450.00 | category: hardware | description: External 4K monitor | status: pending\n5) ID 5 | user: bob | dept: engineering | amount: $320.00 | category: training | description: OAuth 2.0 deep-dive workshop | status: approved\n6) ID 6 | user: bob | dept: engineering | amount: $67.00 | category: meals | description: Team lunch after the migration shipped | status: approved\n\nWould you like to:\n- filter by department or sta

In [6]:
await runner.run_as("dave", "Search for all documents")

[04/14/26 06:21:23] INFO     HTTP Request: POST                                                     ]8;id=398122;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=364683;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭─────────────────────────────────╮
│ [dave] Search for all documents │
╰─────────────────────────────────╯

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=594844;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=224361;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:21:26] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=50815;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=933313;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=276384;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=436010;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

[04/14/26 06:21:29] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=241533;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=312514;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:21:45] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=801288;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=194275;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool             ┃ args        ┃ status ┃ result                                                          ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ search_documents │ {'q': None} │      - │ {'type': 'text', 'text': '{"identity_method": "api_key",        │
│     │                  │             │        │ "identity_detail": "shared service credential, no user          │
│     │                  │             │        │ identity", "allowed_groups": "all", "count": 8, "documents":    │
│     │                  │             │        │ [{"id": 1, "title": "Architecture decision records, Q1",        │
│     │                  │             │        │ "body": "ADR-014: moved authentication to Keycloak. ADR-015:    │
│     │                  │             │        │ adopted O                                                       │
└─────┴──────────────────┴─────────────┴────────┴─────────────────────────────────────────────────────────────────┘

╭───────────────────────────────────────────────── answer ──────────────────────────────────────────────────╮
│ Here are 8 documents I can access for you:                                                                │
│                                                                                                           │
│ 1. ID 1 — Architecture decision records, Q1                                                               │
│    - Body snippet: ADR-014: moved authentication to Keycloak. ADR-015: adopted OPA for policy.            │
│    - Owner: bob                                                                                           │
│    - Access: engineering                                                                                  │
│                                                                                                           │
│ 2. ID 2 — Migration runbook: legacy auth to keycloak                                                      │
│    - Body snippet: Step 1: stand up Keycloak. Step 2: import realm. Step 3: rotate client secrets weekly. │
│    - Owner: alice                                                                                         │
│    - Access: engineering                                                                                  │
│                                                                                                           │
│ 3. ID 3 — Engineering onboarding                                                                          │
│    - Body snippet: Welcome. Read the README, set up your dev env, ping bob in #eng with questions.        │
│    - Owner: bob                                                                                           │
│    - Access: engineering, public                                                                          │
│                                                                                                           │
│ 4. ID 4 — Production incident postmortem 2026-03-12                                                       │
│    - Body snippet: Root cause: stale JWKS cache caused token validation to fail for 18 minutes.           │
│    - Owner: dave                                                                                          │
│    - Access: platform                                                                                     │
│                                                                                                           │
│ 5. ID 5 — On-call rotation rules                                                                          │
│    - Body snippet: Primary: 7d. Secondary: backup. Escalate to dave for sev1.                             │
│    - Owner: dave                                                                                          │
│    - Access: platform                                                                                     │
│                                                                                                           │
│ 6. ID 6 — Compensation bands H1 2026                                                                      │
│    - Body snippet: L3 75-95k, L4 95-130k, L5 130-180k, L6 180-240k. Confidential.                         │
│    - Owner: dave                                                                                          │
│    - Access: admin                                                                                        │
│                                                                                                           │
│ 7. ID 7 — Performance review guidelines                                                                   │
│    - Body snippet: Calibration meeting third week of every quarter. Manager submits drafts 2w prior.      │
│    - Owner: dave                                                                                          │
│    - Access: admin                    

AgentResult(content='Here are 8 documents I can access for you:\n\n1. ID 1 — Architecture decision records, Q1\n   - Body snippet: ADR-014: moved authentication to Keycloak. ADR-015: adopted OPA for policy.\n   - Owner: bob\n   - Access: engineering\n\n2. ID 2 — Migration runbook: legacy auth to keycloak\n   - Body snippet: Step 1: stand up Keycloak. Step 2: import realm. Step 3: rotate client secrets weekly.\n   - Owner: alice\n   - Access: engineering\n\n3. ID 3 — Engineering onboarding\n   - Body snippet: Welcome. Read the README, set up your dev env, ping bob in #eng with questions.\n   - Owner: bob\n   - Access: engineering, public\n\n4. ID 4 — Production incident postmortem 2026-03-12\n   - Body snippet: Root cause: stale JWKS cache caused token validation to fail for 18 minutes.\n   - Owner: dave\n   - Access: platform\n\n5. ID 5 — On-call rotation rules\n   - Body snippet: Primary: 7d. Secondary: backup. Escalate to dave for sev1.\n   - Owner: dave\n   - Access: platform\n\n6. 

## What did the service see?

The punchline: what identity did the backend service actually extract?

In [7]:
await runner.show_service_identity()

                    INFO     HTTP Request: GET http://127.0.0.1:49495/debug/last-request "HTTP/1.1  ]8;id=252994;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=96532;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭──────── expense-service /debug/last-request ─────────╮
│ method:  api_key                                     │
│ user_id: <none>                                      │
│ detail:  shared service credential, no user identity │
│                                                      │
╰──────────────────────────────────────────────────────╯

                    INFO     HTTP Request: GET http://127.0.0.1:49496/debug/last-request "HTTP/1.1  ]8;id=541054;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=180716;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭──────── document-service /debug/last-request ────────╮
│ method:  api_key                                     │
│ user_id: <none>                                      │
│ detail:  shared service credential, no user identity │
│                                                      │
╰──────────────────────────────────────────────────────╯

The service reports method=api_key for all three users, just like pattern 1.

### What changed from pattern 2

This is a step backward at the service level. In pattern 2, the service knew the username and filtered by it. Here, the service is blind again -- it's back to returning everything. The narrowing moved from the service (pattern 2) to the MCP layer (pattern 3), but the service lost all user context in the process.

If dave asks for documents, the service returns ALL documents (including admin-only ones like compensation bands) because it has no identity to filter by. Any filtering you see in the agent's response is from the MCP server's claim-based narrowing or the LLM's interpretation, not the service enforcing access.

### The weakness

If the MCP server were compromised or a prompt injection bypassed the narrowing logic, the service would happily return everything. We need the authz enforcement to move closer to the service.

In [8]:
await runner.stop()

Pattern p03_inline_claim_agent stopped.

[04/14/26 06:21:50] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=581891;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=775911;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       